In [1]:
pkg_libs <- paste(
  "libcurl4-openssl-dev libxml2-dev libssl-dev",
  "libfontconfig1-dev libharfbuzz-dev libfribidi-dev",
  "libpng-dev libtiff5-dev libjpeg-dev"
)
system(paste("apt-get install -y -qq", pkg_libs, "2>/dev/null"))
message("System-level dependencies are ready.")

System-level dependencies are ready.



In [ ]:
setup_bioc <- function(pkgs) {
  BiocManager::install(pkgs, update = FALSE, ask = FALSE)
}

if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager", repos = "https://cloud.r-project.org")

# Differential expression core
setup_bioc("DESeq2")
setup_bioc("apeglm")
setup_bioc("GEOquery")

# Plotting libraries
cran_pkgs <- c("pheatmap", "ggplot2", "ggrepel", "RColorBrewer")
install.packages(cran_pkgs, repos = "https://cloud.r-project.org", quiet = TRUE)

# Functional annotation
setup_bioc(c("org.Hs.eg.db", "clusterProfiler", "enrichplot"))

# Pathway analysis
setup_bioc("ReactomePA")

message("\nAll R packages successfully installed.")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.3 (2026-03-11)

Installing package(s) 'BiocVersion', 'DESeq2'

also installing the dependencies ‘XVector’, ‘formatR’, ‘abind’, ‘SparseArray’, ‘lambda.r’, ‘futile.options’, ‘Seqinfo’, ‘S4Arrays’, ‘DelayedArray’, ‘futile.logger’, ‘snow’, ‘BH’, ‘S4Vectors’, ‘IRanges’, ‘GenomicRanges’, ‘SummarizedExperiment’, ‘BiocGenerics’, ‘Biobase’, ‘BiocParallel’, ‘matrixStats’, ‘locfit’, ‘MatrixGenerics’, ‘RcppArmadillo’


'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cran.rstudio.com

Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.3 (2026-03

In [ ]:
# Mouse genome annotation
if (!requireNamespace("org.Mm.eg.db", quietly = TRUE))
  BiocManager::install("org.Mm.eg.db", update = FALSE, ask = FALSE)

In [ ]:
library(org.Mm.eg.db)
message("Mouse annotation database (org.Mm.eg.db) loaded.")

In [ ]:
pkgs_to_load <- c(
  "DESeq2", "apeglm", "GEOquery",
  "pheatmap", "ggplot2", "ggrepel", "RColorBrewer",
  "org.Mm.eg.db", "clusterProfiler", "enrichplot", "ReactomePA"
)

invisible(lapply(pkgs_to_load, library, character.only = TRUE))

message("\nAll libraries loaded.")
cat("DESeq2 :", as.character(packageVersion("DESeq2")), "\n")
cat("R       :", R.version.string, "\n")

In [ ]:
system2("gunzip", "GSE222445_manuscript.adults.raw.counts.csv.gz")

raw_df   <- read.csv("GSE222445_manuscript.adults.raw.counts.csv",
                     header = TRUE, check.names = FALSE)
gene_ids <- raw_df[, 1]
head(raw_df)

counts_mat          <- as.data.frame(raw_df[, -1])
rownames(counts_mat) <- make.unique(gene_ids)
head(counts_mat)

In [ ]:
gse_obj <- getGEO("GSE222445")
pheno   <- pData(gse_obj[[1]])

sample_info <- data.frame(
  condition = factor(pheno$`treatment:ch1`, levels = c("Control", "Ethanol")),
  rownames  = pheno$title
)
sample_info

In [ ]:
sample_info <- sample_info[order(sample_info$condition), , drop = FALSE]
counts_mat  <- counts_mat[, sample_info$rownames]
counts_mat  <- counts_mat[order(rownames(counts_mat)), ]

head(counts_mat)
sample_info

In [ ]:
  dds_obj <- DESeqDataSetFromMatrix(
  countData = as.matrix(round(counts_mat)), # Ensure counts are integers
  colData   = sample_info,
  design    = ~ condition
)
dds_obj

In [ ]:
row_totals <- rowSums(counts(dds_obj))
dds_obj    <- dds_obj[row_totals >= 10, ]
dds_obj

In [ ]:
# PCA plot
vst_obj <- vst(dds_obj, blind = TRUE)
message("VST complete.")

pca_df   <- plotPCA(vst_obj, intgroup = "condition", returnData = TRUE)
pct_var  <- round(100 * attr(pca_df, "percentVar"))

grp_colors <- c("Control" = "#2166AC", "Ethanol" = "#F4A582")

options(repr.plot.width = 9, repr.plot.height = 7)
p_pca <- ggplot(pca_df, aes(PC1, PC2, color = condition, shape = condition)) +
  geom_point(size = 4, alpha = 0.8) +
  scale_color_manual(values = grp_colors) +
  xlim(c(-12, 10)) + ylim(c(-15, 15)) +
  xlab(paste0("PC1: ", pct_var[1], "% variance")) +
  ylab(paste0("PC2: ", pct_var[2], "% variance")) +
  ggtitle("PCA — Alcohol Use on Mouse (GSE222445)") +
  theme_minimal(base_size = 14) +
  theme(legend.position = "bottom",
        plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_pca)

In [ ]:
# Distance Heatmap
dist_mat <- as.matrix(dist(t(assay(vst_obj))))

annot_df <- data.frame(Condition = sample_info$condition)
rownames(annot_df) <- colnames(counts_mat)

annot_colors <- list(Condition = c("Control" = "#2166AC", "Ethanol" = "#F4A582"))

blue_pal <- colorRampPalette(rev(brewer.pal(9, "Blues")))(255)

options(repr.plot.width = 9, repr.plot.height = 8)
pheatmap(
  dist_mat,
  clustering_distance_rows = dist(t(assay(vst_obj))),
  clustering_distance_cols = dist(t(assay(vst_obj))),
  annotation_col   = annot_df,
  annotation_colors = annot_colors,
  col  = blue_pal,
  main = "Sample-to-Sample Distance Heatmap"
)

In [ ]:
# Fit the DESeq2 negative binomial model
dds_obj <- DESeq(dds_obj)
message("\nDESeq2 model fitting done.")

options(repr.plot.width = 8, repr.plot.height = 6)
plotDispEsts(dds_obj, main = "Dispersion Estimates — Mean vs. Dispersion")

In [ ]:
resultsNames(dds_obj)

de_res <- results(
  dds_obj,
  contrast = c("condition", "Ethanol", "Control"),
  alpha    = 0.1
)

cat("\n  Summary: Ethanol vs Control\n")
cat("══════════════════════════════\n\n")
summary(de_res)

In [ ]:
cat("Available coefficients:\n")
resultsNames(dds_obj)

shrunk_res <- lfcShrink(
  dds_obj,
  coef = "condition_Ethanol_vs_Control",
  type = "apeglm"
)

cat("  Shrunken LFC: Ethanol vs Control\n")
cat("═════════════════════════════════════\n\n")
summary(shrunk_res)

In [ ]:
# MA Plots
options(repr.plot.width = 14, repr.plot.height = 6)
par(mfrow = c(1, 2))
plotMA(de_res,     main = "MA Plot: Ethanol vs Control\n(unshrunk)",       ylim = c(-10, 10))
plotMA(shrunk_res, main = "MA Plot: Ethanol vs Control\n(apeglm shrunk)", ylim = c(-10, 10))
par(mfrow = c(1, 1))

In [ ]:
# Volcano Plot
vlc_df       <- as.data.frame(shrunk_res)
vlc_df$gene  <- rownames(vlc_df)

# Label each gene as Up, Down, or not significant
vlc_df$status <- with(vlc_df, ifelse(
  !is.na(padj) & padj < 0.1 & abs(log2FoldChange) > 1,
  ifelse(log2FoldChange > 1, "Up", "Down"),
  "NS"
))

# Pick the 20 most significant DE genes to label
label_set <- vlc_df[order(vlc_df$padj), ]
label_set <- head(label_set[label_set$status != "NS", ], 20)

options(repr.plot.width = 10, repr.plot.height = 8)
p_volcano <- ggplot(vlc_df, aes(log2FoldChange, -log10(padj))) +
  geom_point(aes(color = status), size = 1.2, alpha = 0.6) +
  scale_color_manual(
    values = c("Up" = "#B2182B", "Down" = "#2166AC", "NS" = "grey70"),
    name   = "Regulation",
    labels = c("Downregulated", "Not Significant", "Upregulated")
  ) +
  geom_vline(xintercept = c(-1, 1), linetype = "dashed", color = "grey40") +
  geom_hline(yintercept = -log10(0.1), linetype = "dashed", color = "grey40") +
  geom_text_repel(data = label_set, aes(label = gene),
                  size = 3, max.overlaps = 20, box.padding = 0.5) +
  labs(x = "Log2 Fold Change", y = "-Log10(Adjusted P-value)",
       title = "Volcano Plot — Ethanol vs Control (Shrunken LFC)") +
  theme_minimal(base_size = 13) +
  theme(legend.position = "bottom",
        plot.title = element_text(hjust = 0.5, face = "bold"))
print(p_volcano)

In [ ]:
# Heatmap of the 30 most significant genes, z-score scaled per gene
top30_genes <- head(rownames(shrunk_res[order(shrunk_res$padj), ]), 30)

expr_mat    <- assay(vst_obj)[top30_genes, ]
expr_scaled <- t(scale(t(expr_mat)))

div_pal <- colorRampPalette(c("#2166AC", "white", "#B2182B"))(100)

options(repr.plot.width = 10, repr.plot.height = 10)
pheatmap(
  expr_scaled,
  annotation_col = annot_df,
  annotation_colors = annot_colors,
  cluster_rows  = TRUE,
  cluster_cols  = TRUE,
  show_rownames = TRUE,
  show_colnames = FALSE,
  fontsize_row  = 8,
  color = div_pal,
  main  = "Top 30 Differential Expression Genes : Z-score Normalized Expression"
)

In [ ]:
# Expression profile of the single most significant gene
best_gene  <- rownames(shrunk_res[order(shrunk_res$padj), ])[1]
cat("Most significant gene:", best_gene, "\n")

cnt_df <- plotCounts(dds_obj, gene = best_gene,
                     intgroup = "condition", returnData = TRUE)

options(repr.plot.width = 7, repr.plot.height = 6)
ggplot(cnt_df, aes(condition, count, fill = condition)) +
  geom_boxplot(alpha = 0.7, outlier.shape = NA) +
  geom_jitter(width = 0.2, size = 2.5, alpha = 0.6) +
  scale_fill_manual(values = c("Control" = "#2166AC", "Ethanol" = "#F4A582")) +
  scale_y_log10() +
  labs(y = "Normalized Count (log10 scale)",
       title = paste0("Expression of ", best_gene, " across conditions")) +
  theme_minimal(base_size = 14) +
  theme(legend.position = "none",
        plot.title = element_text(hjust = 0.5, face = "bold"))

In [ ]:
# Separate up- and down-regulated gene sets
de_hits <- shrunk_res[
  !is.na(shrunk_res$padj) &
  shrunk_res$padj < 0.1   &
  abs(shrunk_res$log2FoldChange) > 1, ]

up_set   <- rownames(de_hits[de_hits$log2FoldChange > 0, ])
down_set <- rownames(de_hits[de_hits$log2FoldChange < 0, ])

cat("Significant DE genes (|LFC| > 1, padj < 0.1):\n")
cat("  Upregulated in Ethanol:  ", length(up_set),   "\n")
cat("  Downregulated in Ethanol:", length(down_set), "\n")

all_de <- c(up_set, down_set)

go_res <- enrichGO(
  gene          = all_de,
  OrgDb         = org.Mm.eg.db,
  keyType       = "SYMBOL",
  ont           = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff  = 0.1,
  qvalueCutoff  = 0.2,
  readable      = TRUE
)

cat("\nTop 10 enriched GO Biological Process terms (Mouse):\n\n")
print(head(go_res, 10))

In [ ]:
# GO enrichment and produce bar + dot plots
de_hits <- shrunk_res[
  !is.na(shrunk_res$padj) &
  shrunk_res$padj < 0.1   &
  abs(shrunk_res$log2FoldChange) > 1, ]

up_set   <- rownames(de_hits[de_hits$log2FoldChange > 0, ])
down_set <- rownames(de_hits[de_hits$log2FoldChange < 0, ])

cat("Significant DE genes (|LFC| > 1, padj < 0.1):\n")
cat("  Upregulated in Ethanol:  ", length(up_set),   "\n")
cat("  Downregulated in Ethanol:", length(down_set), "\n")

all_de <- c(up_set, down_set)

go_res <- enrichGO(
  gene          = all_de,
  OrgDb         = org.Mm.eg.db,
  keyType       = "SYMBOL",
  ont           = "BP",
  pAdjustMethod = "BH",
  pvalueCutoff  = 0.1,
  qvalueCutoff  = 0.2,
  readable      = TRUE
)

cat("\nTop 10 enriched GO Biological Process terms (Mouse):\n\n")
print(head(go_res, 10))

if (!is.null(go_res) && nrow(as.data.frame(go_res)) > 0) {
  n_terms <- nrow(as.data.frame(go_res))
  cat("Total enriched GO terms found:", n_terms, "\n")

  options(repr.plot.width = 10, repr.plot.height = 8)
  print(barplot(go_res, showCategory = 15, title = "Top 15 Enriched GO BP Terms"))
  print(dotplot(go_res, showCategory = 15, title = "GO Enrichment Analysis - Ethanol vs Control"))
} else {
  cat("No enriched GO terms found with current thresholds (padj < 0.1, qvalue < 0.2).\n")
}

In [ ]:
# top GO results as a table
head(as.data.frame(go_res), 15)

In [ ]:
# Enrichment map
if (nrow(as.data.frame(go_res)) >= 5) {
  go_sim <- pairwise_termsim(go_res)
  options(repr.plot.width = 12, repr.plot.height = 10)
  print(emapplot(go_sim, showCategory = 20) +
        ggtitle("Enrichment Map — GO Biological Process"))
} else {
  cat("Not enough GO terms for enrichment map.\n")
}

entrez_map <- bitr(
  all_de,
  fromType = "SYMBOL",
  toType   = "ENTREZID",
  OrgDb    = org.Mm.eg.db
)

reactome_res <- enrichPathway(
  gene          = entrez_map$ENTREZID,
  organism      = "mouse",
  pAdjustMethod = "BH",
  pvalueCutoff  = 0.1,
  readable      = TRUE
)

if (!is.null(reactome_res) && nrow(as.data.frame(reactome_res)) > 0) {
  cat("Top Reactome Pathways (Mouse):\n")
  cols_show <- c("Description", "GeneRatio", "p.adjust", "Count")
  print(head(as.data.frame(reactome_res)[, cols_show], 10))
  options(repr.plot.width = 10, repr.plot.height = 8)
  print(dotplot(reactome_res, showCategory = 15,
                title = "Reactome Pathway Enrichment — Mouse"))
} else {
  cat("No significant Reactome pathways found at padj < 0.1.\n")
  cat("This is common in specific mouse models — the GO results remain our primary analysis.\n")
}

In [ ]:
sorted_res <- shrunk_res[order(shrunk_res$padj), ]

f_full <- "DESeq2_Ethanol_vs_Control_full_results.csv"
f_sig  <- "DESeq2_Ethanol_significant_genes_padj01.csv"

write.csv(as.data.frame(sorted_res), file = f_full, row.names = TRUE)

sig_df <- as.data.frame(sorted_res[
  !is.na(sorted_res$padj) & sorted_res$padj < 0.1, ])

write.csv(sig_df, file = f_sig, row.names = TRUE)

cat("\u2713 Full results \u2192", f_full, "\n")
cat("\u2713 Significant genes \u2192", f_sig, "\n")
cat("  Total significant (padj < 0.1):", nrow(sig_df), "\n")